# Readme



# Imports

In [1]:
import pandas as pd
import os # for csv export

# for url html content parsing
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [4]:
df = pd.read_csv("../data/16-19_test.csv")

print(f"Loaded {len(df)} cases")
df.head()

Loaded 2488 cases


,id,indexed_date,language,media_name,media_url,publish_date,title,url
0,d0cb0ceda597ae6eb3e5c7e013d3a2276b9f24cabec598...,2025-07-24 02:06:58.254299+00:00,zh,chinanews.com,chinanews.com,2018-06-30,乘火车出行，“军人依法优先”政策有哪些？-中新网,http://www.chinanews.com/mil/2018/06-30/855212...
1,42ade47c5a72eee37d476c3b8252d60ebdab2e0e4f61dd...,2025-07-24 01:55:34.596230+00:00,zh,xinhuanet.com,xinhuanet.com,2018-06-30,长汀：客家祠堂里永传的“星火”-新华网,http://www.xinhuanet.com/politics/2018-06/30/c...
2,23e96912bbb5d8b168cbe391102387d3d5d31f88b510dc...,2025-07-24 01:55:34.190754+00:00,zh,xinhuanet.com,xinhuanet.com,2018-06-30,乘火车出行，“军人依法优先”政策有哪些？-新华网,http://www.xinhuanet.com/politics/2018-06/30/c...
3,7778c2031b728d74af28e40d0cdcf400d6011f6acf61f9...,2025-07-24 01:53:56.315790+00:00,zh,voc.com.cn,voc.com.cn,2018-06-30,47年党龄老党员的心愿――“交最后一次党费”,http://hunan.voc.com.cn/article/201806/2018063...
4,e81d0152513ec6b05bbd320a05283df504e6a4f56dbe10...,2025-07-24 01:52:19.195323+00:00,zh,people.com.cn,people.com.cn,2018-06-30,比球技，更要比文采?--体育--人民网,http://sports.people.com.cn/worldcup2018/n1/20...


2488

In [6]:
# keep only relevant data.
df_pretty = df[ ["title", "publish_date", "media_name", "url"] ].copy()
df_pretty.head()

,title,publish_date,media_name,url
0,乘火车出行，“军人依法优先”政策有哪些？-中新网,2018-06-30,chinanews.com,http://www.chinanews.com/mil/2018/06-30/855212...
1,长汀：客家祠堂里永传的“星火”-新华网,2018-06-30,xinhuanet.com,http://www.xinhuanet.com/politics/2018-06/30/c...
2,乘火车出行，“军人依法优先”政策有哪些？-新华网,2018-06-30,xinhuanet.com,http://www.xinhuanet.com/politics/2018-06/30/c...
3,47年党龄老党员的心愿――“交最后一次党费”,2018-06-30,voc.com.cn,http://hunan.voc.com.cn/article/201806/2018063...
4,比球技，更要比文采?--体育--人民网,2018-06-30,people.com.cn,http://sports.people.com.cn/worldcup2018/n1/20...


# Prep Data for Media Textbody Identification

get one url per media outlet

In [7]:
df_pretty.media_name.unique()
f"number of unique medias is {len(df_pretty.media_name.unique())}"

array(['chinanews.com', 'xinhuanet.com', 'voc.com.cn', 'people.com.cn',
       'ce.cn', 'huanqiu.com', 'lzbs.com.cn', '01ny.cn', 'cnhubei.com',
       'syd.com.cn', 'bjd.com.cn', 'hangzhou.com.cn', 'lyd.com.cn',
       'southcn.com', 'china.com.cn', 'chinadaily.com.cn', 'xhby.net',
       'yangtse.com', 'cctv.com', 'gmw.cn', 'cet.com.cn', 'qstheory.cn',
       'zjol.com.cn', 'cri.cn', 'ycwb.com', 'dzwww.com', 'rednet.cn',
       'qianlong.com', 'legaldaily.com.cn', 'jcrb.com', 'timedg.com',
       'xmnn.cn', 'jxnews.com.cn', '66wz.com', 'qlwb.com.cn', 'yunnan.cn',
       'nmgnews.com.cn', 'cnnb.com.cn', 'fynews.net', 'cjn.cn',
       'eastday.com', '81.cn'], dtype=object)

'number of unique medias is 42'

In [12]:
# create new df that only keeps one sample url per media compnay.
    
df_unique_media_urls = df_pretty.drop_duplicates(subset = "media_name")[["media_name", "url"]].reset_index(drop = True) # erases original index number
df_unique_media_urls

,media_name,url
0,chinanews.com,http://www.chinanews.com/mil/2018/06-30/855212...
1,xinhuanet.com,http://www.xinhuanet.com/politics/2018-06/30/c...
2,voc.com.cn,http://hunan.voc.com.cn/article/201806/2018063...
3,people.com.cn,http://sports.people.com.cn/worldcup2018/n1/20...
4,ce.cn,http://ce.cn/xwzx/gnsz/gdxw/201806/30/t2018063...
5,huanqiu.com,http://world.huanqiu.com/exclusive/2018-07/123...
6,lzbs.com.cn,http://www.lzbs.com.cn/ttnews/2018-07/02/conte...
7,01ny.cn,http://news.01ny.cn/2018/fazhizongheng_0713/96...
8,cnhubei.com,http://sports.cnhubei.com/2018/0716/399532.shtml
9,syd.com.cn,http://finance.syd.com.cn/system/2018/07/30/01...


In [21]:
# check if result df contains the same number of media outlets as the original df.
df_unique_media_urls.shape[0] == df_pretty.media_name.unique().shape[0]

True

In [25]:
# save csv to processing in get_textbody_div.ipynb

df_unique_media_urls.to_csv("../data/unique_media_urls.csv", index = False)
print(f"output CSV file saved to {os.getcwd()}")

output CSV file saved to /Users/wyx/Desktop/26X/code


# Filter
visit each url using the "media_body_tags.csv".
check that media content must contain keyword 退役运动员

In [10]:
KEYWORD = "退役运动员"
df_checker = pd.read_csv("../data/media_textbody_tags.csv", index_col = 0) # designate column 0 as index. otherwise pandas treats it as a variable column.
df_checker

,media_name,body_div
0,chinanews.com,div.left_zw
1,xinhuanet.com,div#p-detail
2,voc.com.cn,div#content
3,people.com.cn,div.rm_txt_con.cf
7,01ny.cn,div.hide_me.P.H
10,bjd.com.cn,div.bjd-not-found__con
11,hangzhou.com.cn,td.Dlife_bcolor02
12,lyd.com.cn,div.page-bottom-word
14,china.com.cn,div.articleBody
15,chinadaily.com.cn,div.article


In [11]:
# test the formula that gets the correct textbody tag according to the given url's media

for url in df.loc[:2, "url"]:
    match = df_checker[ df_checker.media_name.apply(lambda name: name in url) ].body_div.values
    print(match)

['div.left_zw']
['div#p-detail']
['div#p-detail']


In [23]:
# df_goodmedia: filter df_pretty so that it only contains urls from media sources with identified textbody tag. 

df_goodmedia = df_pretty[
    df_pretty.apply(lambda r: df_checker.media_name.apply(lambda name: name in r.url).any(), # returns true if any match is found.
                    axis = 1)].copy()

df_goodmedia.shape[0]
df_goodmedia.sample(5)

2340

,title,publish_date,media_name,url
549,女子链球再度摘金夺银&nbsp;罗娜、王峥扛起大旗,2018-08-25,01ny.cn,http://news.01ny.cn/2018/tiyuxinwen_0825/99432...
883,株洲市荷塘区打造“复转军人之家”营造军人社会尊崇氛围-新华网,2018-11-06,xinhuanet.com,http://www.xinhuanet.com/local/2018-11/06/c_12...
1572,跳“空中芭蕾”的中国女孩-新华网,2019-01-04,xinhuanet.com,http://www.xinhuanet.com/local/2019-01/04/c_11...
2287,洛阳市委十一届十次全会精神在全市各地持续引发强烈反响,2019-06-07,lyd.com.cn,http://news.lyd.com.cn/system/2019/06/07/03135...
2167,湖南开展“洞庭风雷”专项行动守护河湖安澜,2019-04-09,voc.com.cn,http://hnrb.voc.com.cn/article/201904/20190409...


In [93]:
# RUN all urls and check textbody for KEYWORD. saving result to keyword_check = []

def make_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

def contains_keyword(driver, url):
    try:
        driver.get(url)
        time.sleep(1)

        textbody_tag = df_checker[ df_checker.media_name.apply(lambda name: name in url) ].body_div.values[0] # .values return list. need str i.e. first and only element in the list
        
        content = driver.find_element(By.CSS_SELECTOR, textbody_tag)
        if (KEYWORD in content.text):
            return True
        else:
            return False
            
    except Exception as e:
        print(f"Failed: {url} -> {e}")
        return False

driver = make_driver()
keyword_check = []

try:
    for i, url in enumerate(df_goodmedia["url"]):
        if(contains_keyword(driver, str(url))):
            keyword_check.append(1)
        else:
            keyword_check.append(0)
        print(f"i = {i}, checked url {url}. result = {keyword_check[i]}.\n")
finally:
    driver.quit()

i = 0, checked url http://www.chinanews.com/mil/2018/06-30/8552129.shtml. result = 0.

i = 1, checked url http://www.xinhuanet.com/politics/2018-06/30/c_1123059151.htm. result = 0.

i = 2, checked url http://www.xinhuanet.com/politics/2018-06/30/c_1123057836.htm. result = 0.

i = 3, checked url http://hunan.voc.com.cn/article/201806/201806301027556256.html. result = 0.

i = 4, checked url http://sports.people.com.cn/worldcup2018/n1/2018/0630/c418684-30097237.html. result = 0.

i = 5, checked url http://sports.people.com.cn/worldcup2018/n1/2018/0630/c418684-30097222.html. result = 0.

Failed: http://sports.people.com.cn/worldcup2018/n1/2018/0630/c418684-30097195.html -> Message: no such element: Unable to locate element: {"method":"css selector","selector":"div.rm_txt_con.cf"}
  (Session info: chrome=151.0.7922.75); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
0   chromedriver   

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [97]:
# save keyword check result as new column in df_goodmedia. then filter to url that conatins keyword.

df_goodmedia["contains_keyword"] = keyword_check

df_filtered = df_goodmedia[ df_goodmedia.contains_keyword == 1 ]
df_filtered
df_filtered.to_csv("../data/actual_athlete_media.csv")

,title,publish_date,media_name,url,contains_keyword
33,广东中山：2020年全市将有超303个足球场,2018-07-03,chinanews.com,http://www.chinanews.com/sh/2018/07-03/8555102...,1
54,“体操皇后”霍尔金娜：在“有限”里造出个“无限”来,2018-07-09,people.com.cn,http://sports.people.com.cn/n1/2018/0709/c2215...,1


In [99]:
# clickable 
df_filtered.url.values

array(['http://www.chinanews.com/sh/2018/07-03/8555102.shtml',
       'http://sports.people.com.cn/n1/2018/0709/c22155-30134403.html'],
      dtype=object)